# Creating SEV, NPS, OF
_Author: ..._<br>
_Created: Sep. 5, 2025_<br>
_Last updated: Sep. 5, 2025 by ..._

---

This tutorial ... *short description*

How exactly one builds those objects is a matter of taste. We present the two main approaches but of course you can individualize the workflow and/or combine the two methods.

## Method 1: Using `cait.versatile` (recommended)

... show how histograms can be inspected, how cuts can be defined using logical numpy array, how iterators for selected events are created and finally, how SEV, NPS, OF are built. Add small notes for common pitfalls.

### SEV

In [ ]:
# ... show easy quality cuts and how they are determined using vai.Histogram
quality_cuts = ai.cuts.LogicalCut()
quality_cuts.add_condition((dh["events/onset", 0]>-2.63)*(dh["events/onset", 0]<-2.51)); print(quality_cuts.counts())
quality_cuts.add_condition((dh["events/rise_time", 0]>1.4)*(dh["events/rise_time", 0]<1.6)); print(quality_cuts.counts())

In [ ]:
# Show how iterator can be used to assess quality of pulses for SEV
sev_events = dh.get_event_iterator(
    "events", 
    0, 
    flag=quality_cuts.get_flag()
).with_processing(vai.RemoveBaseline())

vai.Preview(sev_events)

In [ ]:
# Discuss how 'SEV refinement' can be achieved

# Create a preliminary SEV and fit it to contributing pulses
sev = vai.SEV(sev_events)

*_, rms = vai.apply(vai.TemplateFit(sev), sev_events)

# Remove RMS outliers and build refined SEV.
# Also subtract linear baseline fit this time.
sev = vai.SEV(sev_events[:, rms<0.0005].with_processing(vai.RemoveBaseline(dict(model="exp", where=1/8))))

sev.show()

In [ ]:
# ... saving
sev.to_file(...)
sev.to_dh(...)

In [ ]:
# Also show how one could produce a fit

# Remove remaining noise by fitting template
fit_slice = slice(int(0.24*len(sev)), int(0.6*len(sev)))
t_for_fit = sev.t[fit_slice]
sev_for_fit = sev[fit_slice]

# Find optimal parameters
pars, _  = sp.optimize.curve_fit(ai.fit.pulse_template, 
                                 t_for_fit, 
                                 sev_for_fit, 
                                 p0=[-1, -0.5, 1.5, 1, 0.1, 10], 
                                 bounds=((-10, -10, -10, 0, 0, 0), (10, 10, 10, np.inf, np.inf, np.inf)))
# Evaluate model using optimal parameters
sev_fit = ai.fit.pulse_template(sev.t, *pars)
scale = np.max(sev_fit)

t, A, T = sev.t, np.zeros_like(sev), np.zeros_like(sev)
cond = t > pars[0]
A[cond] =  pars[1] * (np.exp(-(t[cond] - pars[0]) / pars[3]) - np.exp(-(t[cond] - pars[0]) / pars[4])) / scale
T[cond] =  pars[2] * (np.exp(-(t[cond] - pars[0]) / pars[5]) - np.exp(-(t[cond] - pars[0]) / pars[3])) / scale

# Plot result
vai.Line({
    "SEV": [t, sev], 
    f"Fit\ntau_n={pars[-3]:.2f} ms\ntau_in={pars[-2]:.2f} ms\ntau_t={pars[-1]:.2f} ms": [t, sev_fit],
    "Athermal component": [t, A],
    "Thermal component": [t, T],
},
    xlabel="Time (ms)",
);

### NPS

In [ ]:
# performing quality cuts again
noise_cuts = ai.cuts.LogicalCut()
noise_cuts.add_condition(dh["noise/pulse_height", 1]<0.001); print(noise_cuts.counts())
# ...

In [ ]:
# Inspecting quality of baselines
noise_events = dh.get_event_iterator(
    "noise", 
    channel=1, 
    flag=noise_cuts.get_flag()
).with_processing([
    vai.RemoveBaseline({"model": 3, "where": 1.0}),
    vai.TukeyWindow(),
])

vai.Preview(noise_events)

In [ ]:
# Show this very powerful cut (remove everything that has large RMS when fit with cubic baselines)
_, fit_rms = vai.apply(vai.FitBaseline(model=3, where=1.0), noise_events.with_batchsize(10))
vai.Histogram(fit_rms)

In [ ]:
nps = vai.NPS(noise_events[:, (fit_rms.flatten()>0.000284)*(fit_rms.flatten()<0.000336)])

In [ ]:
# Show how to save

### OF

In [ ]:
# For of = vai.OF(sev, nps) it's probably more important to discuss some pitfalls

## Method 2: Using `cait.VizTool`

... show that one can also use the vizTool (this tutorial can be shorter imo, but we should mention it). We can also only explain it for SEV, because NPS would work analogously. OF is trivial anyways.

## Estimating baseline resolution
Regardless of how you built SEV, NPS, and the OF, you will most likely use the filter to estimate the baseline resolution next. This can be easily achieved as follows:

In [ ]:
noise_traces_for_sim = noise_events[:, ::50]

simulate_ph = 0.01 # V

of = vai.OF.from_file("files/...")
sev = vai.SEV.from_file("files/...")

# Simulate fixed pulse height on noise traces
sim_events = vai.iterators.PulseSimIterator(
    noise_traces_for_sim, 
    pulse_heights=simulate_ph*np.ones(len(noise_traces_for_sim)),
    sev=sev,
)

# Reconstruct pulse heights
ph_rec = vai.apply(
    vai.OFPulseHeight(of, sev, max_search=(2040, 2060)), 
    sim_events.with_batchsize(10)
)[0]

In [ ]:
# Plot distribution

min_fit, max_fit = simulate_ph*1000*0.95, simulate_ph*1000*1.04
phs = 1000*ph_rec
fit_x = np.linspace(min_fit, max_fit, 100)

hist = vai.Histogram(
    phs,
    xlabel="Reconstructed OF PH (mV)", 
    bins=fit_x, 
)

gauss_fitpar = sp.stats.norm.fit(phs[(phs>min_fit)*(phs<max_fit)])
hist.add_line(x=fit_x, 
              y=len(phs[(phs>min_fit)*(phs<max_fit)])*np.diff(fit_x)[0]*sp.stats.norm.pdf(fit_x, *gauss_fitpar), 
              name=f"μ={gauss_fitpar[0]:.3f} mV\nσ={gauss_fitpar[1]:.3f} mV")

## Tips

... if you can think of any tips/tricks that may be useful, we can collect them here. E.g. explain how one can use `sev.to_file` and `sev.to_dh` etc. to organize those objects.

Another thing we could add here is how one would fit a standard event to get its parameters

## Advanced

.. show how one can use ``vai.apply`` to come up with more parameters on the fly.